# LLaVA-Interleave (0.5B) — Experiment A

Same prompt wording as `Phase5_Qwen_ExperimentA.ipynb` (system prompt, support-example framing, test-image instructions are textually identical), adapted to LLaVA's message API. This is what makes a later Qwen-vs-LLaVA comparison fair.

**Read the smoke-test cell (Section 8) output carefully before running the full loop** — this is the first time this prompt structure has been tried against this model.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

## 2. Back up any existing LLaVA Experiment A results before overwriting

In [ ]:
import os, shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["llava_experiment_A.csv", "llava_experiment_A_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

## 3. Install packages
LLaVA-Interleave only needs `transformers >= 4.35.3`, far less version-sensitive than Qwen2.5-VL was — but upgrading anyway to get the current `apply_chat_template` / image-text-to-text conventions.

In [ ]:
!pip install -q -U transformers accelerate

## 4. Load the model

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_NAME = "llava-hf/llava-interleave-qwen-0.5b-hf"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Loading model...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("✅ Model loaded successfully!")

## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Load an image and resize it to reduce memory usage."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

## 6. Prompt builder — Experiment A
Text is identical to `build_messages()` in `Phase5_Qwen_ExperimentA.ipynb`. The API shape differs from Qwen: LLaVA's conversation turns reference images with a placeholder (`{"type": "image"}`), and the actual PIL images are passed separately, in the same order the placeholders appear, via `images=` to the processor — not embedded inside the message dicts the way Qwen2.5-VL did.

In [ ]:
def build_messages_A(sample, support_images, test_image):
    """
    Build the LLaVA conversation for one test image — Experiment A.
    Returns (conversation, images) — images is an ordered list matching
    each {"type": "image"} placeholder in the conversation.
    """
    conversation = []
    images = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will first receive five labelled support examples.

Each support example contains:
- one image
- the exact crowd count
- the correct crowd density

Learn from these examples.

Then you will receive one test image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    for i in range(1, 6):
        count = int(sample[f"support{i}_count"])
        density = int(sample[f"support{i}_density"])

        conversation.append({
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": f"""Support Example {i}

Study this labelled support example carefully.
"""}
            ]
        })
        images.append(load_image(support_images[i - 1]))

        conversation.append({
            "role": "assistant",
            "content": [{
                "type": "text",
                "text": f"""Ground Truth

Count: {count}

Density: {density} ({DENSITY_TEXT[density]})
"""
            }]
        })

    conversation.append({
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })
    images.append(load_image(test_image))

    return conversation, images

## 7. Inference and parsing

In [ ]:
import gc
import re
import torch


def run_llava(conversation, images):
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(images=images, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device, torch.float16) if v.dtype.is_floating_point else v.to(model.device)
              for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)

    trimmed_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    response = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    del inputs, generated_ids, trimmed_ids
    gc.collect()
    torch.cuda.empty_cache()

    return response.strip()


def parse_response(response):
    """Same robust parser used for the final Qwen runs."""
    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        count = int(m.group(1))

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

## 8. Smoke test — READ THIS OUTPUT BEFORE RUNNING THE FULL LOOP
Prints the actual rendered prompt text and the raw model response, and surfaces any error clearly rather than letting it crash deep inside the full run.

In [ ]:
import pandas as pd
import os

ROOT = "/content/Phase5_Input"
TRAIN_DIR = f"{ROOT}/train/images"
TEST_DIR = f"{ROOT}/test/images"

support_df = pd.read_csv(f"{ROOT}/support_mapping.csv")
test_df = pd.read_csv(f"{ROOT}/test.csv")

sample0 = support_df.iloc[0]
test_image0 = os.path.join(TEST_DIR, f"{int(sample0['test_filename']):04d}.jpg")
support_images0 = [
    os.path.join(TRAIN_DIR, f"{int(sample0[f'support{i}_filename']):04d}.jpg")
    for i in range(1, 6)
]

conversation0, images0 = build_messages_A(sample0, support_images0, test_image0)

try:
    prompt_text0 = processor.apply_chat_template(conversation0, add_generation_prompt=True)
    print("=== RENDERED PROMPT (first 2000 chars) ===")
    print(prompt_text0[:2000])
    print("...")
    print()
    print(f"Number of images passed: {len(images0)}")
    print()

    response0 = run_llava(conversation0, images0)
    print("=== MODEL RESPONSE ===")
    print(repr(response0))
    print()
    print("Parsed:", parse_response(response0))

except Exception as e:
    print("❌ SMOKE TEST FAILED — do not run the full loop yet.")
    print(f"{type(e).__name__}: {e}")
    print()
    print("Common fixes if this is a chat-template error:")
    print("  - Try changing the system message's \"content\" from a list of dicts to a plain string")
    print("  - Check processor.apply_chat_template(conversation0, add_generation_prompt=True, tokenize=False)")
    print("    to see the raw template output and compare against the model card's example")
    raise

## 9. Full run — all 1600 test images, saved continuously to Drive

In [ ]:
from tqdm import tqdm
import os
import gc
import time
import torch

OUT_PATH = os.path.join(RESULT_DIR, "llava_experiment_A.csv")

if "results_A" not in globals():
    results_A = []

done_filenames = {r["test_filename"] for r in results_A}
remaining = support_df[~support_df["test_filename"].astype(int).isin(done_filenames)]
print(f"Already done: {len(done_filenames)} | Remaining: {len(remaining)}")

MAX_RETRIES_PER_IMAGE = 2

for _, sample in tqdm(remaining.iterrows(), total=len(remaining)):

    test_image = os.path.join(TEST_DIR, f"{int(sample['test_filename']):04d}.jpg")
    support_images = [
        os.path.join(TRAIN_DIR, f"{int(sample[f'support{i}_filename']):04d}.jpg")
        for i in range(1, 6)
    ]

    conversation, images = build_messages_A(sample, support_images, test_image)

    response = None
    for attempt in range(1, MAX_RETRIES_PER_IMAGE + 1):
        try:
            response = run_llava(conversation, images)
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"\n[OOM] test_filename={int(sample['test_filename'])} "
                  f"attempt {attempt}/{MAX_RETRIES_PER_IMAGE} — clearing cache and retrying...")
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(2)

    if response is None:
        pred_count, pred_density = None, None
        response = "ERROR: CUDA out of memory after retries"
    else:
        pred_count, pred_density = parse_response(response)

    results_A.append({
        "test_filename": int(sample["test_filename"]),
        "predicted_count": pred_count,
        "predicted_density": pred_density,
        "raw_response": response
    })

    pd.DataFrame(results_A).to_csv(OUT_PATH, index=False)

failed = sum(1 for r in results_A if r["predicted_count"] is None)
print("Finished:", len(results_A), f"({failed} failed after retries)")
print("Saved to:", OUT_PATH)

## 10. Evaluate — counting metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

llava_A = pd.DataFrame(results_A)

llava_A_eval = llava_A.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", llava_A_eval["total_count"].isna().sum())

y_true_count = llava_A_eval["total_count"]
y_pred_count = llava_A_eval["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = llava_A_eval["total_count"] > 0
mape = np.mean(np.abs(
    (llava_A_eval.loc[nonzero, "total_count"] - llava_A_eval.loc[nonzero, "predicted_count"])
    / llava_A_eval.loc[nonzero, "total_count"]
)) * 100

print("LLaVA Experiment A — Counting")
print("==============================")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")

## 11. Evaluate — density classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true_density = llava_A_eval["crowd_density"]
y_pred_density = llava_A_eval["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("LLaVA Experiment A — Density")
print("=============================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

## 12. Save evaluated results to Drive

In [ ]:
EVAL_PATH = os.path.join(RESULT_DIR, "llava_experiment_A_evaluated.csv")
llava_A_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(llava_A_eval), "rows")
print("Path:", EVAL_PATH)